# Aura Graph Analytics: GraphSAGE embedding training via sessions

<a target="_blank" href="https://colab.research.google.com/github/neo4j/graph-data-science-client/blob/main/examples/examples/graphsage-unsupervised-via-session.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

This Jupyter notebook is hosted [here](https://github.com/neo4j/graph-data-science-client/blob/main/examples/examples/graphsage-unsupervised-via-session.ipynb) in the Neo4j Graph Data Science Client Github repository.

The notebook shows how to use the `graphdatascience` Python library to train an unsupervised
GraphSAGE model on Cora and compute node embeddings on an Aura Graph Analytics (AGA) Session,
using the typed `gds.graph_sage.unsupervised` endpoints.

## Prerequisites

This notebook requires having the Aura Graph Analytics [feature](https://neo4j.com/docs/aura/graph-analytics/#aura-gds-serverless) enabled for your Neo4j Aura project, and a session backed by the **Python runtime**.

You also need to have the `graphdatascience` Python library installed, version `2.0a1` or later.

In [ ]:
%pip install "graphdatascience>=2.0a1" python-dotenv matplotlib numpy

## Credentials

In [ ]:
import os

from dotenv import load_dotenv

# This allows to load required secrets from `.env` file in local directory
load_dotenv(".env")

## Get or Create Session

## Aura API credentials

In [ ]:
from graphdatascience.session import AuraAPICredentials, GdsSessions

api_credentials = AuraAPICredentials(
    client_id=os.environ["CLIENT_ID"],
    client_secret=os.environ["CLIENT_SECRET"],
    # If your account is a member of several projects, you must also specify the project ID to use
    project_id=os.environ.get("PROJECT_ID", None),
)

sessions = GdsSessions(api_credentials=api_credentials)

## Creating a new session

A new session is created by calling `sessions.get_or_create()` with a representative name, the
session memory, the cloud location, and a time-to-live (TTL), which ensures that the session is
automatically deleted after being unused for the set time, to avoid incurring costs.

In [ ]:
from datetime import timedelta

from graphdatascience.session import CloudLocation, SessionMemory

# Explicitly define the size of the session
memory = SessionMemory.m_4GB

# Specify your cloud location
cloud_location = CloudLocation("gcp", "europe-west1")

# Create a GDS session!
gds = sessions.get_or_create(
    # we give it a representative name
    session_name="graphsage-embeddings",
    memory=memory,
    ttl=timedelta(minutes=30),
    cloud_location=cloud_location,
)

# Verify the connectivity. Hints towards TLS or firewall issues if this fails directly after get_or_create
gds.verify_connectivity()

## Load Cora

The Cora citation dataset ships with the client and is loaded directly into the session, with `CITES`
relationships treated as **undirected** so the random walks used by unsupervised GraphSAGE can
traverse citations in both directions.

In [ ]:
gds.graph.drop("cora", fail_if_missing=False)
G = gds.graph.datasets.load_cora(undirected=True)
print("Loaded graph:", G.name())

## Train GraphSAGE (Unsupervised)

`gds.graph_sage.unsupervised.train` submits the training job, blocks until completion (with progress
reporting), and returns the trained model together with a train summary result.
`featureProperties` selects the node properties used as input features (here Cora's `features` vector).
`nodeLabels` and `relationshipTypes` default to `*` (project everything).

In [ ]:
modelname = "gs_cora_unsup_typed"

In [ ]:
model, train_result = gds.graph_sage.unsupervised.train(
    G,
    model_name=modelname,
    feature_properties=["features"],
    epochs=10,
    embedding_dimension=64,
    learning_rate=0.001,
    num_neighbors=[25, 10],
    activation_function="relu",
    aggregator="mean",
    num_walks=10,
    walk_depth=3,
    random_seed=42,
)

print("Trained model:", model.name())

In [ ]:
train_result

## Run Inference

`gds.graph_sage.unsupervised.stream` runs the predict job against the stored model and streams the results.
Unsupervised GraphSAGE produces a node embedding per node (rather than class predictions).
Inference projects the same `featureProperties` the model was trained on.

In [ ]:
embeddings_df = gds.graph_sage.unsupervised.stream(
    G,
    model_name=model.name(),
    feature_properties=["features"],
    random_seed=42,
)

print("Columns:", list(embeddings_df.columns))
embeddings_df

## Visualize embeddings

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# colour each paper by its `subject` to see how embeddings cluster
subjects = gds.graph.node_properties.stream(G, "subject")
embeddings_df = embeddings_df.merge(subjects, on="nodeId")

# project the 64-dim embeddings to 2D with an SVD (no extra dependencies needed)
X = np.stack(embeddings_df["embedding"].to_numpy())
X = X - X.mean(axis=0)
U, _, _ = np.linalg.svd(X, full_matrices=False)
embeddings_df["x"], embeddings_df["y"] = U[:, 0], U[:, 1]

fig, ax = plt.subplots(figsize=(8, 6))
for subject in sorted(embeddings_df["subject"].unique()):
    part = embeddings_df[embeddings_df["subject"] == subject]
    ax.scatter(part["x"], part["y"], s=25, label=f"subject {subject}")
ax.set_title("GraphSAGE embeddings of Cora papers")
ax.set_xlabel("component 1")
ax.set_ylabel("component 2")
ax.legend()
fig.tight_layout()
plt.show()

## (Optional) cleanup

In [ ]:
gds.model.delete(modelname, fail_if_missing=False)
sessions.delete(session_name="graphsage-embeddings")